### To generate standard datasets
#### task demo gift box

### Step 1. convert segment ploygen points to rle type

In [ ]:
import os
from pathlib import Path
os.chdir("../../")

print(os.getcwd())
import json
from pycocotools import mask as mask_utils

ori_annotation_path = Path("datasets/giftbox_task/annotations/annotations_0001.json")
## need to create file
rle_type_file_path = Path("datasets/giftbox_task/rle_annotations/annotations_0001.json")


def rle_convert(ori_annotation_path, rle_type_file_path):
    #rle_type_file_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(ori_annotation_path, 'r') as f:
        coco_data = json.load(f)

    images_map = {img['id']: img for img in coco_data['images']}
    converted_count = 0
    for ann in coco_data['annotations']:
        seg = ann.get('segmentation', [])
        
        if not seg or (isinstance(seg, dict) and 'counts' in seg and isinstance(seg['counts'], str)):
            continue
        
        img_info = images_map[ann['image_id']]
        height, width = img_info['height'], img_info['width']
        try:
            rles = mask_utils.frPyObjects(seg, height, width)
            rle = mask_utils.merge(rles)
            if isinstance(rle['counts'], bytes):
                rle['counts'] = rle['counts'].decode('utf-8')
            ann['segmentation'] = rle
            converted_count += 1
            
        except Exception as e:
            print(f"Warning: 转换标注 ID {ann.get('id')} 失败: {e}")
            continue
    
    with open(rle_type_file_path, 'w') as f:
        json.dump(coco_data, f, indent=2)
    
    print(f"{converted_count} save: {rle_type_file_path}")
    


In [ ]:
rle_convert(ori_annotation_path,rle_type_file_path)



In [ ]:
## batch convert 
from tqdm import tqdm

origin_anno_dir = Path("datasets/giftbox_task/annotations")
rle_anno_dir = Path("datasets/giftbox_task/rle_annotations")

f_name = origin_anno_dir.glob("*.json")

for name in tqdm(f_name,desc="convert process:"):
    o_file = origin_anno_dir / name.name
    rle_file = rle_anno_dir / name.name

    rle_convert(o_file,rle_file)



## Step 2. make standard datasets structure

In [ ]:
## eg:

"""
"images":
[
    {
      "id": 0,
      "video_id":"0001",  
      "file_name": "1/video1/metaclip_1_1001_c122868928880ae52b33fae1.jpeg",
      "frame_id" : 1 ,
      "width": 600,
      "height": 600,
      "text_inputs": ["red box", "green small box", "small zip", "toy car"],
      "queried_categories": ["0", "2001", "0", "0"],
      "is_instance_exhaustive": 1,
      "is_pixel_exhaustive": 1,
      "scene_description" : "box free and closed",
      "task_type": "assembly task"
    },
]

"annotation":
[
   {
      "id": 1,
      "image_id": 0,
      "text_idx": 0,
      "source": "manual",
      "area": 0.00248,
      "bbox": [0.44333, 0.0, 0.10833, 0.05833],
      "segmentation": {
       ##use rle type "counts": "`kk42fb01O1O1O1O001O1O1O001O1O00001O1O001O001O0000000000O1001000O010O02O001N10001N0100000O10O1000O10O010O100O1O1O1O1O0000001O0O2O1N2N2Nobm4",
        "size": [600, 600]
      },
      "category_id": 1,
      "iscrowd": 0
    },
]
"""

In [ ]:

label_dir = Path("datasets/giftbox_task/label")
import numpy as np


def get_description(video_id,frame_idx):

    des_dict = {
    "1": "box free and closed",
    "2": "box fixed and closed",
    "3": "box fixed and opening",
    "4": "box open and toy free",
    "5": "toy in hand and outside box",
    "6": "toy in box and box opened",
    "7": "toy in box and box closing",
    "8": "toy in box and box on table",
    "9": "right hand holding red bag, left hand holding green box",
    "10": "toy in box and box in hand",
    "11": "gift has been packaged"
}   
    label_f = label_dir / (str(video_id)+"_label.txt")


    matrix = np.loadtxt(label_f, dtype=int)

    cls_id = str(matrix[frame_idx][1])

    return des_dict[cls_id]




In [ ]:
def get_instance_annostatus(image_id,annotation_list,categorys):

    text_inputs,queried_categories = [],[]

    
    return text_inputs,queried_categories


In [ ]:
def build_annotation_field(image_id,annotation_list):
    pass

In [ ]:
## first build images json 
import re


rle_annotation_file = Path("datasets/giftbox_task/rle_annotations/annotations_0001.json")

def build_image_field(rle_annotation_file):

    with open(ori_annotation_path, 'r') as f:
        coco_data = json.load(f)

    ## image_id    
    images_list = { img for img in coco_data['images']}
    annotation_list = { ann for ann in coco_data['annotations']}
    categorys = {cate for cate in coco_data['categories']}

    match = re.search(r"annotations_(\d+)\.json",rle_annotation_file.name)

    if match:
        video_id = match.group(1)
    else:
        video_id = -1

    target_image_fieid = []
    target_anno_fieid = []

    target_dict = {
        "id": 0,
        "video_id": str(video_id),  
        "file_name": "1/video1/metaclip_1_1001_c122868928880ae52b33fae1.jpeg",
        "frame_idx" : 1 ,
        "width": 640,
        "height": 480,
        "text_inputs": ["red box", "green small box", "small zip", "toy car"], ## 类别名称
        "queried_categories": ["0", "1", "2", "3"],   ## 查询类别编号
        "is_instance_exhaustive": 1,   ## 实例标注是否穷尽
        "is_pixel_exhaustive": 1,      ## 像素级标注是否穷尽
        "scene_description" : "box free and closed",
        "task_type": "assembly task"   ## stable
    }

    target_anno = {
    "id": 1,    ### anno_id
    "image_id": 0,
    "source": "automatic",
    "area": 0.00248,
    "bboxs": [[0.44333, 0.0, 0.10833, 0.05833],
            [0.44333, 0.0, 0.10833, 0.05833]],
    "segmentations": [{
    ##use rle type 
    "counts": "`kk42fb01O1O1O1O001O1O1O001O1O00001O1O001O001O0000000000O10",
    "size": [600, 600]
    },{
    "counts": "`kk42fb01O1O1O1O001O1O1O001O1O00001O1O001O001O0000000000O10",
    "size": [600, 600]
    }],
    "queried_categories": ["0", "1", "2", "3"],   ## 查询类别编号
    "iscrowd": 0
    }

    
    for i_field in images_list:

        target_dict["id"] = i_field["id"]
        target_dict["file_name"] = str(video_id) + "/" + i_field["file_name"]
        target_dict["frame_idx"] = i_field["frame_idx"]
        target_dict["width"] = i_field["width"]
        target_dict["height"] = i_field["height"]

        match_frameid = re.search(r"frame_(\d+)\.jpg", i_field["frame_idx"])
        if match_frameid:
            frame_idx = match_frameid.group(1)
        else:
            frame_idx = -1

        target_dict["scene_description"] = get_description(int(video_id),int(frame_idx))

        text_inputs,queried_categories = get_instance_annostatus(i_field["id"],annotation_list,categorys)

        target_dict["text_inputs"] = text_inputs
        target_dict["queried_categories"] = queried_categories 

        target_image_fieid.append(target_dict)

        build_annotation_field(i_field["id"],annotation_list)

